In [12]:
import torch
import torch.nn as nn
import torch.functional as F
import transformers
from transformers import (
    DataCollatorForSeq2Seq, 
    T5ForConditionalGeneration, 
    T5Tokenizer,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, TaskType

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
running on cuda


In [15]:
model_name = "google-t5/t5-base"

dataset = load_dataset("kaanrkaraman/code2doc")

tokenizer = T5Tokenizer.from_pretrained(model_name)

def preprocess(examples):
    inputs = ["Summarize: " + x for x in examples["function_code"]]
    targets = examples["documentation"]
    
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)


lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,                    # rank — lower = less VRAM, less capacity
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],  # T5 attention projections; this is the standard choice
)

model = T5ForConditionalGeneration.from_pretrained(model_name)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() 
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Map:   0%|          | 0/10684 [00:00<?, ? examples/s]

Map:   0%|          | 0/1340 [00:00<?, ? examples/s]

Map:   0%|          | 0/1334 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 223,788,288 || trainable%: 0.3953


In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
input_ids = tokenizer("Summarize: def hello(arg): print(arg)", return_tensors="pt").to(model.device)

output = model.generate(**input_ids, cache_implementation="static")
print(tokenizer.decode(output[0], skip_special_tokens=True))

/home/park/proj/lib/python3.10/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


: hello(arg): hello(arg): hello(arg): print(arg
